In [17]:
import logging

import bm25s
import pandas as pd

from beir import LoggingHandler
from beir.retrieval import models
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch

In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

## Data Loading

In [3]:
corpus, queries, qrels = GenericDataLoader(data_folder="../data/beir", qrels_file=f"../data/beir/qrels.tsv").load_custom()

2025-03-28 16:16:46 - Loading Corpus...


  0%|          | 0/1499 [00:00<?, ?it/s]

100%|██████████| 1499/1499 [00:00<00:00, 84508.48it/s]

2025-03-28 16:16:46 - Loaded 1499 Documents.
2025-03-28 16:16:46 - Doc Example: {'text': 'bismillah yuga lampah balukar janglar meunang kahayang balukar sasar pedar ringkang sing jembar sabar tong jadi hambar ihtér raksa rasa ukir pikir mun gering tong rungsing mun cageur tong badeur ulah ceurik jadi jalma leutik ulah sombong abong di gedong lain batur kabéh gelé dulur néang halal awur amal ulah lieur ku madu dunya jung sanding ka nu agung gusti wéti ngarti nu sajati pariksa ati', 'title': 'JEMBAR SABAR'}
2025-03-28 16:16:46 - Loading Queries...
2025-03-28 16:16:46 - Loaded 7491 Queries.
2025-03-28 16:16:46 - Query Example: apa maksud dari bismillah yuga lampah


## Evaluate using Sentence Transformers

In [4]:
model = DenseRetrievalExactSearch(models.SentenceBERT("multi-qa-mpnet-base-dot-v1"), batch_size=16)
model

2025-03-28 16:17:01 - Use pytorch device_name: cuda
2025-03-28 16:17:01 - Load pretrained SentenceTransformer: multi-qa-mpnet-base-dot-v1


In [5]:
retriever = EvaluateRetrieval(model, score_function="dot")  # dot or cos_sim
results = retriever.retrieve(corpus, queries)

2025-03-28 16:17:30 - Encoding Queries...


Batches: 100%|██████████| 469/469 [00:08<00:00, 53.40it/s]


2025-03-28 16:17:39 - Sorting Corpus by document length (Longest first)...
2025-03-28 16:17:39 - Encoding Corpus in batches... Warning: This might take a while!
2025-03-28 16:17:39 - Scoring Function: Dot Product (dot)
2025-03-28 16:17:39 - Encoding Batch 1/1...


Batches: 100%|██████████| 94/94 [00:20<00:00,  4.51it/s]


In [6]:
#### Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K  where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, retriever.k_values)

2025-03-28 16:18:21 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - NDCG@1: 0.1236
2025-03-28 16:18:24 - NDCG@3: 0.1653
2025-03-28 16:18:24 - NDCG@5: 0.1778
2025-03-28 16:18:24 - NDCG@10: 0.1933
2025-03-28 16:18:24 - NDCG@100: 0.2376
2025-03-28 16:18:24 - NDCG@1000: 0.2876
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - MAP@1: 0.1236
2025-03-28 16:18:24 - MAP@3: 0.1549
2025-03-28 16:18:24 - MAP@5: 0.1618
2025-03-28 16:18:24 - MAP@10: 0.1682
2025-03-28 16:18:24 - MAP@100: 0.1761
2025-03-28 16:18:24 - MAP@1000: 0.1776
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - Recall@1: 0.1236
2025-03-28 16:18:24 - Recall@3: 0.1954
2025-03-28 16:18:24 - Recall@5: 0.2256
2025-03-28 16:18:24 - Recall@10: 0.2738
2025-03-28 16:18:24 - Recall@100: 0.4954
2025-03-28 16:18:24 - Recall@1000: 0.9146
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - P@1: 0.1236
2025-03-28 16:18:24

In [18]:
metrics = [{"metric": k.split("@")[0], "k": k.split("@")[1], "value": v} for col in [ndcg, _map, recall, precision] for k, v in col.items()]
pd.DataFrame(metrics)

,metric,k,value
0,NDCG,1,0.12362
1,NDCG,3,0.16533
2,NDCG,5,0.17777
3,NDCG,10,0.19328
4,NDCG,100,0.23762
5,NDCG,1000,0.28762
6,MAP,1,0.12362
7,MAP,3,0.15494
8,MAP,5,0.16185
9,MAP,10,0.16821


## Evaluate using BM25

Source: https://www.kaggle.com/code/xhlulu/benchmark-bm25-on-beir

In [25]:
def postprocess_results_for_eval(results, scores, query_ids):
    """
    Given the queried results and scores output by BM25S, postprocess them
    to be compatible with BEIR evaluation functions.
    query_ids is a list of query ids in the same order as the results.
    """

    results_record = [
        {"id": qid, "hits": results[i], "scores": list(scores[i])}
        for i, qid in enumerate(query_ids)
    ]

    result_dict_for_eval = {
        res["id"]: {
            docid: float(score) for docid, score in zip(res["hits"], res["scores"])
        }
        for res in results_record
    }

    return result_dict_for_eval

In [16]:
corpus_ids, corpus_lst = [], []
for key, val in corpus.items():
    corpus_ids.append(key)
    corpus_lst.append(val["title"] + " " + val["text"])

qids, queries_lst = [], []
for key, val in queries.items():
    qids.append(key)
    queries_lst.append(val)

In [22]:
corpus_tokens = bm25s.tokenize(corpus_lst)
query_tokens = bm25s.tokenize(queries_lst)

In [23]:
model = bm25s.BM25(method="lucene", k1=1.2, b=0.75)
model.index(corpus_tokens)

2024-12-28 16:48:09 - Building index from IDs objects


In [24]:
queried_results, queried_scores = model.retrieve(
    query_tokens, corpus=corpus_ids, k=1000, n_threads=1
)

In [26]:
results_dict = postprocess_results_for_eval(queried_results, queried_scores, qids)

In [28]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels, results_dict, [1, 3, 5, 10, 100, 1000]
)

2024-12-28 16:50:23 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - NDCG@1: 0.5321
2024-12-28 16:50:26 - NDCG@3: 0.6313
2024-12-28 16:50:26 - NDCG@5: 0.6512
2024-12-28 16:50:26 - NDCG@10: 0.6704
2024-12-28 16:50:26 - NDCG@100: 0.6951
2024-12-28 16:50:26 - NDCG@1000: 0.7023
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - MAP@1: 0.5321
2024-12-28 16:50:26 - MAP@3: 0.6078
2024-12-28 16:50:26 - MAP@5: 0.6189
2024-12-28 16:50:26 - MAP@10: 0.6269
2024-12-28 16:50:26 - MAP@100: 0.6320
2024-12-28 16:50:26 - MAP@1000: 0.6322
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - Recall@1: 0.5321
2024-12-28 16:50:26 - Recall@3: 0.6990
2024-12-28 16:50:26 - Recall@5: 0.7470
2024-12-28 16:50:26 - Recall@10: 0.8061
2024-12-28 16:50:26 - Recall@100: 0.9219
2024-12-28 16:50:26 - Recall@1000: 0.9805
2024-12-28 16:50:26 - 

2024-12-28 16:50:26 - P@1: 0.5321
2024-12-28 16:50:26